# EXP 08 — Tournament-Specific Strong Ensemble (Row-Wise, Leakage-Safe, AW-MAE-Oriented)

Notebook ini membangun pipeline prediksi `team_goals` dan `opp_goals` dengan fokus pada:

1. **domain-specific modeling** berbasis `tournament`,
2. **row-wise training** untuk setiap baris observasi,
3. **strong ensemble** antar model tabular,
4. **post-processing** yang dipilih berdasarkan **official AW-MAE validation**,
5. **final inference** pada `test.csv` dan pembuatan file submission:
   **`exp08_tournament_specific_strong_ensemble.csv`**

## Catatan penting
- `ground_truth_bersih.csv` / file ground-truth sejenis **tidak digunakan** untuk training, tuning, feature engineering, calibration, evaluasi validation, maupun inference final.
- Evaluasi resmi notebook ini dilakukan **sekali per pertandingan** (`match_id`) agar tidak menghitung dua baris yang merepresentasikan match yang sama secara ganda.
- Notebook ini sengaja memakai fitur yang **tersedia konsisten di train dan test**, lalu menambahkan **fitur turunan aman** dari kolom umum.

In [ ]:
# =========================
# 00. Setup dan Konfigurasi
# =========================

import os
import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -------------------------
# Konfigurasi eksperimen
# -------------------------
EXPERIMENT_NAME = "exp08_tournament_specific_strong_ensemble"
FINAL_CSV_NAME = f"{EXPERIMENT_NAME}.csv"

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

VALID_FRAC = 0.20
MIN_DOMAIN_ROWS = 2000        # threshold explicit untuk special tournament models
MAX_SPECIAL_TOURNAMENTS = 8
MIN_ROWS_FOR_TRAINING = 300   # guardrail agar subset sangat kecil tidak dipaksa dilatih
PLOT_TOP_N_TOURNAMENTS = 15

POSTPROCESS_GRID = {
    "total_scale": [1.00, 0.97, 0.94],
    "draw_shrink": [0.00, 0.05, 0.10, 0.15],
}

# Catatan:
# - Threshold domain dibuat cukup tinggi agar model khusus hanya dipakai pada tournament
#   yang datanya relatif stabil.
# - Nilai ini bisa diubah, tetapi di notebook ini dibuat eksplisit dan terkontrol.

## 00.1 Deteksi library modeling

Eksperimen utama mencoba tiga keluarga model tabular:

- `CatBoostRegressor`
- `LightGBMRegressor`
- `XGBoostRegressor` (jika environment mendukung)

Jika ada library yang tidak tersedia atau training gagal, notebook tetap **runnable**
dengan fallback elegan ke model yang berhasil dilatih.

In [ ]:
AVAILABLE_FAMILIES = {}
IMPORT_ERRORS = {}

try:
    from catboost import CatBoostRegressor
    AVAILABLE_FAMILIES["cat"] = True
except Exception as e:
    AVAILABLE_FAMILIES["cat"] = False
    IMPORT_ERRORS["cat"] = repr(e)

try:
    from lightgbm import LGBMRegressor
    AVAILABLE_FAMILIES["lgb"] = True
except Exception as e:
    AVAILABLE_FAMILIES["lgb"] = False
    IMPORT_ERRORS["lgb"] = repr(e)

try:
    from xgboost import XGBRegressor
    AVAILABLE_FAMILIES["xgb"] = True
except Exception as e:
    AVAILABLE_FAMILIES["xgb"] = False
    IMPORT_ERRORS["xgb"] = repr(e)

available_model_families = [k for k, v in AVAILABLE_FAMILIES.items() if v]

print("Available model families:", available_model_families)
if IMPORT_ERRORS:
    print("\nImport errors (jika ada):")
    for k, v in IMPORT_ERRORS.items():
        print(f"- {k}: {v}")

# 01. Load Data

Pada eksperimen ini:

- dataset training utama adalah `train.csv`
- inference final dilakukan pada `test.csv`
- target adalah `team_goals` dan `opp_goals`
- struktur data adalah **2 baris per pertandingan**

In [ ]:
train_df_raw = pd.read_csv(TRAIN_PATH)
test_df_raw = pd.read_csv(TEST_PATH)

print("train shape:", train_df_raw.shape)
print("test  shape:", test_df_raw.shape)

display(train_df_raw.head(3))
display(test_df_raw.head(3))

required_train_cols = {"Id", "match_id", "date", "tournament", "team_goals", "opp_goals"}
required_test_cols = {"Id", "match_id", "date", "tournament"}

assert required_train_cols.issubset(train_df_raw.columns), "Kolom train wajib belum lengkap."
assert required_test_cols.issubset(test_df_raw.columns), "Kolom test wajib belum lengkap."

In [ ]:
# Parse date lebih awal agar aman untuk split time-based
train_df_raw["date"] = pd.to_datetime(train_df_raw["date"], errors="coerce")
test_df_raw["date"] = pd.to_datetime(test_df_raw["date"], errors="coerce")

assert train_df_raw["date"].notna().all(), "Ada date train yang gagal diparse."
assert test_df_raw["date"].notna().all(), "Ada date test yang gagal diparse."

train_df_raw = train_df_raw.sort_values(["date", "match_id", "Id"]).reset_index(drop=True)
test_df_raw = test_df_raw.sort_values(["date", "match_id", "Id"]).reset_index(drop=True)

print("Train date range:", train_df_raw["date"].min(), "->", train_df_raw["date"].max())
print("Test  date range:", test_df_raw["date"].min(), "->", test_df_raw["date"].max())

# 02. EDA Singkat yang Relevan

Fokus EDA di eksperimen ini:

1. memverifikasi struktur **2 baris per pertandingan**,
2. melihat distribusi `tournament`,
3. mengidentifikasi kategori yang sangat besar vs sangat jarang,
4. memastikan eksperimen domain split masuk akal.

In [ ]:
# -------------------------
# Struktur match: berapa baris per match_id?
# -------------------------
rows_per_match = train_df_raw.groupby("match_id").size().value_counts().sort_index()
print("Distribusi jumlah baris per match_id (train):")
display(rows_per_match.to_frame("n_match_ids"))

# sanity check utama
pct_two_rows = (
    (train_df_raw.groupby("match_id").size() == 2).mean() * 100
)
print(f"Persentase match dengan tepat 2 baris: {pct_two_rows:.2f}%")

In [ ]:
# -------------------------
# Analisis tournament
# -------------------------
train_tournament_counts = train_df_raw["tournament"].fillna("MISSING").value_counts()
test_tournament_counts = test_df_raw["tournament"].fillna("MISSING").value_counts()

print("Jumlah kategori tournament unik di train:", train_tournament_counts.shape[0])
print("Jumlah kategori tournament unik di test :", test_tournament_counts.shape[0])

display(train_tournament_counts.head(20).rename("train_rows").to_frame())
display(test_tournament_counts.head(20).rename("test_rows").to_frame())

largest_tournaments = train_tournament_counts.head(10)
rare_tournaments = train_tournament_counts[train_tournament_counts <= 50].sort_values()

print("Tournament terbesar di train:")
display(largest_tournaments.to_frame("train_rows"))

print("Contoh tournament sangat jarang di train (<= 50 rows):")
display(rare_tournaments.head(20).to_frame("train_rows"))

In [ ]:
# Visualisasi frekuensi tournament teratas
plot_top = train_tournament_counts.head(PLOT_TOP_N_TOURNAMENTS).sort_values()
plt.figure(figsize=(10, 6))
plot_top.plot(kind="barh")
plt.title("Top Tournament di Train (berdasarkan jumlah row)")
plt.xlabel("Jumlah row")
plt.ylabel("Tournament")
plt.tight_layout()
plt.show()

# 03. Validation Strategy Anti-Leakage

Strategi validasi yang dipakai:

- **time-based split** berbasis `date`
- seluruh baris dengan `match_id` yang sama dijaga tetap berada pada split yang sama
- validasi dihitung **sekali per match**, bukan dua kali untuk dua row

Dengan kata lain, kita meniru situasi inference final:
model belajar dari pertandingan yang lebih lama, lalu dievaluasi pada pertandingan yang lebih baru.

In [ ]:
def temporal_group_split(df: pd.DataFrame, valid_frac: float = 0.20):
    match_dates = (
        df.groupby("match_id", as_index=False)["date"]
        .min()
        .sort_values("date")
        .reset_index(drop=True)
    )
    n_valid = max(1, int(np.ceil(len(match_dates) * valid_frac)))
    valid_match_ids = set(match_dates.iloc[-n_valid:]["match_id"])
    is_valid = df["match_id"].isin(valid_match_ids)

    train_part = df.loc[~is_valid].copy().reset_index(drop=True)
    valid_part = df.loc[is_valid].copy().reset_index(drop=True)
    return train_part, valid_part

train_fold_raw, valid_fold_raw = temporal_group_split(train_df_raw, valid_frac=VALID_FRAC)

print("train_fold shape:", train_fold_raw.shape)
print("valid_fold shape:", valid_fold_raw.shape)
print("train_fold max date:", train_fold_raw["date"].max())
print("valid_fold min date:", valid_fold_raw["date"].min())

# leakage audit sederhana
train_match_ids = set(train_fold_raw["match_id"].unique())
valid_match_ids = set(valid_fold_raw["match_id"].unique())

assert train_match_ids.isdisjoint(valid_match_ids), "Ada match_id yang bocor antara train dan validation."
print("Leakage audit match_id: AMAN")

# 04. Official AW-MAE Evaluator

Di bawah ini kita implementasikan evaluator offline yang mengikuti aturan resmi:

- Base MAE per match
- Penalti exact / outcome / goal-difference
- Outcome multiplier
- Non-linear power
- Tournament weighting

Poin penting:
karena data train bersifat **row-wise 2 baris per match**, maka prediksi row-level akan
diubah dulu menjadi representasi **canonical per match_id**, lalu baru dihitung skornya **sekali per pertandingan**.

In [ ]:
# ==================================================
# 04. Official AW-MAE evaluator dan helper scoring
# ==================================================

EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50

def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()

    if "fifa world cup" in t and "qualification" not in t:
        return 2.00

    if (
        "afc championship" in t
        or "afc asian cup" in t
        or t == "asian cup"
        or " asian cup" in t
    ):
        return 1.80

    if "friendly" in t:
        return 0.96

    return 1.20


def outcome_label(a: int, b: int) -> int:
    if a > b:
        return 1
    if a < b:
        return -1
    return 0


def build_side_index(df_rows: pd.DataFrame) -> pd.DataFrame:
    temp = df_rows.copy()
    temp = temp.sort_values(["match_id", "Id"]).reset_index(drop=True)
    temp["side_idx"] = temp.groupby("match_id").cumcount()
    return temp


def rows_to_canonical_match_df(
    df_rows: pd.DataFrame,
    pred_team_cont: np.ndarray,
    pred_opp_cont: np.ndarray,
    include_truth: bool = True,
) -> pd.DataFrame:
    temp = build_side_index(df_rows)
    temp["pred_team_cont"] = np.asarray(pred_team_cont, dtype=float)
    temp["pred_opp_cont"] = np.asarray(pred_opp_cont, dtype=float)

    records = []
    for match_id, g in temp.groupby("match_id", sort=False):
        g = g.sort_values("side_idx").reset_index(drop=True)

        tournament = g.loc[0, "tournament"]

        if len(g) >= 2:
            pred_a = float(np.mean([g.loc[0, "pred_team_cont"], g.loc[1, "pred_opp_cont"]]))
            pred_b = float(np.mean([g.loc[0, "pred_opp_cont"], g.loc[1, "pred_team_cont"]]))
        else:
            pred_a = float(g.loc[0, "pred_team_cont"])
            pred_b = float(g.loc[0, "pred_opp_cont"])

        row = {
            "match_id": match_id,
            "tournament": tournament,
            "pred_team_cont": pred_a,
            "pred_opp_cont": pred_b,
        }

        if include_truth:
            row["team_goals_true"] = int(g.loc[0, "team_goals"])
            row["opp_goals_true"] = int(g.loc[0, "opp_goals"])

        records.append(row)

    return pd.DataFrame(records)


def postprocess_match_predictions(
    match_df: pd.DataFrame,
    total_scale: float = 1.0,
    draw_shrink: float = 0.0,
) -> pd.DataFrame:
    out = match_df.copy()

    a = np.clip(out["pred_team_cont"].to_numpy(dtype=float), 0, None) * float(total_scale)
    b = np.clip(out["pred_opp_cont"].to_numpy(dtype=float), 0, None) * float(total_scale)

    if draw_shrink > 0:
        mu = (a + b) / 2.0
        a = mu + (a - mu) * (1.0 - float(draw_shrink))
        b = mu + (b - mu) * (1.0 - float(draw_shrink))

    out["team_goals_pred"] = np.rint(np.clip(a, 0, None)).astype(int)
    out["opp_goals_pred"] = np.rint(np.clip(b, 0, None)).astype(int)
    return out


def evaluate_awmae_from_match_df(match_pred_df: pd.DataFrame) -> dict:
    y_team = match_pred_df["team_goals_true"].astype(int).to_numpy()
    y_opp = match_pred_df["opp_goals_true"].astype(int).to_numpy()
    p_team = match_pred_df["team_goals_pred"].astype(int).to_numpy()
    p_opp = match_pred_df["opp_goals_pred"].astype(int).to_numpy()
    tournaments = match_pred_df["tournament"].astype(str).to_numpy()

    weights = np.array([get_tournament_weight(t) for t in tournaments], dtype=float)

    base_mae = (np.abs(y_team - p_team) + np.abs(y_opp - p_opp)) / 2.0
    exact = ((y_team == p_team) & (y_opp == p_opp)).astype(int)
    outcome = np.array(
        [int(outcome_label(a, b) == outcome_label(c, d)) for a, b, c, d in zip(y_team, y_opp, p_team, p_opp)],
        dtype=int,
    )
    gd = (((y_team - y_opp) == (p_team - p_opp))).astype(int)

    penalty = (
        EXACT_PENALTY * (1 - exact)
        + OUTCOME_PENALTY * (1 - outcome)
        + GD_PENALTY * (1 - gd)
    )

    multiplier = np.where(outcome == 1, 1.0, WRONG_OUTCOME_MULTIPLIER)
    raw_loss = base_mae + penalty
    loss = np.power(raw_loss * multiplier, NONLINEAR_POWER)

    return {
        "awmae": float(np.sum(loss * weights) / np.sum(weights)),
        "base_mae": float(np.mean(base_mae)),
        "exact_rate": float(np.mean(exact)),
        "outcome_acc": float(np.mean(outcome)),
        "gd_acc": float(np.mean(gd)),
        "n_matches": int(len(match_pred_df)),
    }


def metrics_from_row_predictions(
    df_rows: pd.DataFrame,
    pred_team_cont: np.ndarray,
    pred_opp_cont: np.ndarray,
    total_scale: float = 1.0,
    draw_shrink: float = 0.0,
):
    match_df = rows_to_canonical_match_df(
        df_rows=df_rows,
        pred_team_cont=pred_team_cont,
        pred_opp_cont=pred_opp_cont,
        include_truth=True,
    )
    match_df_pp = postprocess_match_predictions(
        match_df,
        total_scale=total_scale,
        draw_shrink=draw_shrink,
    )
    metrics = evaluate_awmae_from_match_df(match_df_pp)
    return match_df, match_df_pp, metrics


def match_loss_table(match_pred_df: pd.DataFrame) -> pd.DataFrame:
    df = match_pred_df.copy()

    y_team = df["team_goals_true"].astype(int).to_numpy()
    y_opp = df["opp_goals_true"].astype(int).to_numpy()
    p_team = df["team_goals_pred"].astype(int).to_numpy()
    p_opp = df["opp_goals_pred"].astype(int).to_numpy()

    base_mae = (np.abs(y_team - p_team) + np.abs(y_opp - p_opp)) / 2.0
    exact = ((y_team == p_team) & (y_opp == p_opp)).astype(int)
    outcome = np.array(
        [int(outcome_label(a, b) == outcome_label(c, d)) for a, b, c, d in zip(y_team, y_opp, p_team, p_opp)],
        dtype=int,
    )
    gd = (((y_team - y_opp) == (p_team - p_opp))).astype(int)

    penalty = (
        EXACT_PENALTY * (1 - exact)
        + OUTCOME_PENALTY * (1 - outcome)
        + GD_PENALTY * (1 - gd)
    )
    multiplier = np.where(outcome == 1, 1.0, WRONG_OUTCOME_MULTIPLIER)
    raw_loss = base_mae + penalty
    loss = np.power(raw_loss * multiplier, NONLINEAR_POWER)

    df["match_base_mae"] = base_mae
    df["exact"] = exact
    df["outcome_correct"] = outcome
    df["gd_correct"] = gd
    df["raw_loss"] = raw_loss
    df["loss"] = loss
    df["weight"] = df["tournament"].map(get_tournament_weight)
    df["weighted_loss"] = df["loss"] * df["weight"]
    return df

# 05. Preprocessing dan Feature Engineering

Karena `test.csv` tidak memiliki semua fitur historis yang ada di `train.csv`,
fitur utama eksperimen ini dibatasi ke:

1. **kolom yang ada di train dan test**,
2. **fitur turunan aman** dari kolom umum tersebut.

Eksperimen ini **tidak menambahkan rolling baru**.
Fokusnya memang pada:
- split domain per `tournament`,
- strong ensemble,
- post-processing berbasis AW-MAE.

In [ ]:
# ===================================
# 05. Feature engineering yang aman
# ===================================

def add_safe_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Parse / sort assumptions
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Sentinel handling yang aman
    if "altitude_venue" in df.columns:
        df["altitude_venue"] = df["altitude_venue"].replace(-9999, np.nan)

    # Date features
    df["year"] = df["date"].dt.year.astype("float64")
    df["month"] = df["date"].dt.month.astype("float64")
    df["dayofweek"] = df["date"].dt.dayofweek.astype("float64")
    df["dayofyear"] = df["date"].dt.dayofyear.astype("float64")
    df["quarter"] = df["date"].dt.quarter.astype("float64")

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)
    df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 366.0)
    df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 366.0)

    # Numeric interactions aman
    df["population_ratio_log"] = np.log1p(df["population_team"]) - np.log1p(df["population_opp"])
    df["gdp_ratio_log"] = np.log1p(df["gdp_per_capita_team"]) - np.log1p(df["gdp_per_capita_opp"])
    df["travel_diff"] = df["distance_travel_team"] - df["distance_travel_opp"]
    df["travel_sum"] = df["distance_travel_team"] + df["distance_travel_opp"]

    # Binary helpers
    df["same_confed"] = (
        df["confederation_team"].fillna("UNK") == df["confederation_opp"].fillna("UNK")
    ).astype(int)

    df["venue_is_team_country"] = (
        df["venue_country"].fillna("UNK") == df["team"].fillna("UNK")
    ).astype(int)

    df["venue_is_opp_country"] = (
        df["venue_country"].fillna("UNK") == df["opponent"].fillna("UNK")
    ).astype(int)

    t = df["tournament"].fillna("").astype(str).str.lower()
    df["is_friendly_like"] = t.str.contains("friendly").astype(int)
    df["is_qualification_like"] = t.str.contains("qualification").astype(int)
    df["is_world_cup_like"] = t.str.contains("world cup").astype(int)

    return df


train_df = add_safe_features(train_df_raw)
test_df = add_safe_features(test_df_raw)
train_fold = add_safe_features(train_fold_raw)
valid_fold = add_safe_features(valid_fold_raw)

# Fitur dasar hanya yang tersedia di train dan test
common_raw_cols = [c for c in train_df.columns if c in test_df.columns]

# Kolom yang tidak boleh dipakai sebagai fitur
forbidden_feature_cols = {"Id", "match_id", "date"}

feature_cols = [c for c in common_raw_cols if c not in forbidden_feature_cols]

numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_feature_cols = [c for c in feature_cols if c not in numeric_feature_cols]

print("Jumlah feature final:", len(feature_cols))
print("Jumlah numeric feature:", len(numeric_feature_cols))
print("Jumlah categorical feature:", len(categorical_feature_cols))

display(pd.DataFrame({
    "feature_type": (["numeric"] * len(numeric_feature_cols)) + (["categorical"] * len(categorical_feature_cols)),
    "feature_name": numeric_feature_cols + categorical_feature_cols
}))

## 05.1 Ringkasan strategi domain split

Karena kategori `tournament` sangat banyak dan tidak seimbang, kita **tidak**
membuat model khusus untuk semua kategori mentah.

Strategi yang dipakai:

- tournament dengan jumlah row train-fold **>= `MIN_DOMAIN_ROWS`** → dibuatkan **model khusus**
- sisanya → masuk bucket **`OTHER_RARE`**
- tetap disediakan **global fallback model**
- untuk bucket rare / unseen, kita bandingkan:
  - fallback ke **global**
  - fallback ke **OTHER_RARE model**

In [ ]:
def get_special_tournaments(train_part: pd.DataFrame, min_domain_rows: int, max_special_tournaments: int):
    counts = train_part["tournament"].fillna("MISSING").value_counts()
    specials = counts[counts >= min_domain_rows].index.tolist()
    specials = specials[:max_special_tournaments]
    return specials, counts


def assign_tournament_bucket(series: pd.Series, special_tournaments):
    special_set = set(special_tournaments)
    return series.fillna("MISSING").apply(lambda x: x if x in special_set else "OTHER_RARE")


special_tournaments_fold, tournament_counts_fold = get_special_tournaments(
    train_fold,
    min_domain_rows=MIN_DOMAIN_ROWS,
    max_special_tournaments=MAX_SPECIAL_TOURNAMENTS,
)

train_fold["tournament_bucket"] = assign_tournament_bucket(train_fold["tournament"], special_tournaments_fold)
valid_fold["tournament_bucket"] = assign_tournament_bucket(valid_fold["tournament"], special_tournaments_fold)

print("Special tournaments berdasarkan training fold:")
print(special_tournaments_fold)

domain_summary_fold = (
    pd.DataFrame({"train_rows": train_fold["tournament"].fillna("MISSING").value_counts()})
    .join(pd.DataFrame({"valid_rows": valid_fold["tournament"].fillna("MISSING").value_counts()}), how="outer")
    .fillna(0)
    .reset_index()
    .rename(columns={"index": "tournament"})
)

domain_summary_fold["train_rows"] = domain_summary_fold["train_rows"].astype(int)
domain_summary_fold["valid_rows"] = domain_summary_fold["valid_rows"].astype(int)
domain_summary_fold["is_special_model"] = domain_summary_fold["tournament"].isin(special_tournaments_fold)
domain_summary_fold["bucket_if_not_special"] = np.where(
    domain_summary_fold["is_special_model"], domain_summary_fold["tournament"], "OTHER_RARE"
)

display(domain_summary_fold.sort_values(["is_special_model", "train_rows"], ascending=[False, False]).head(30))

# 06. Modeling Helpers

Di bawah ini kita siapkan helper untuk:

- preprocessing,
- training model per family,
- training global / rare / special-domain models,
- prediksi validation dengan fallback policy,
- blend tuning berbasis AW-MAE.

In [ ]:
# ==========================================
# 06. Helper preprocessing, training, infer
# ==========================================

def make_preprocessor():
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                ]),
                numeric_feature_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                ]),
                categorical_feature_cols,
            ),
        ],
        remainder="drop",
    )
    return preprocessor


def instantiate_model(family: str):
    if family == "cat":
        return CatBoostRegressor(
            loss_function="RMSE",
            depth=6,
            learning_rate=0.06,
            iterations=100,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
        )

    if family == "lgb":
        return LGBMRegressor(
            objective="regression",
            n_estimators=140,
            learning_rate=0.05,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED,
            n_jobs=-1,
        )

    if family == "xgb":
        return XGBRegressor(
            objective="reg:squarederror",
            n_estimators=140,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED,
            n_jobs=4,
            tree_method="hist",
        )

    raise ValueError(f"Unknown family: {family}")


def safe_fit_family_bundle(df_train_subset: pd.DataFrame, family: str):
    if family not in available_model_families:
        return None

    if len(df_train_subset) < MIN_ROWS_FOR_TRAINING:
        return None

    try:
        preprocessor = make_preprocessor()
        X_train = preprocessor.fit_transform(df_train_subset[feature_cols])

        model_team = instantiate_model(family)
        model_opp = instantiate_model(family)

        model_team.fit(X_train, df_train_subset["team_goals"].values)
        model_opp.fit(X_train, df_train_subset["opp_goals"].values)

        bundle = {
            "family": family,
            "preprocessor": preprocessor,
            "team_model": model_team,
            "opp_model": model_opp,
            "feature_names_after_preprocess": numeric_feature_cols + categorical_feature_cols,
            "n_rows": len(df_train_subset),
        }
        return bundle

    except Exception as e:
        print(f"[WARN] training failed for family={family}: {repr(e)}")
        return None


def predict_bundle(bundle, df_part: pd.DataFrame):
    X = bundle["preprocessor"].transform(df_part[feature_cols])
    pred_team = bundle["team_model"].predict(X)
    pred_opp = bundle["opp_model"].predict(X)
    return np.asarray(pred_team, dtype=float), np.asarray(pred_opp, dtype=float)


def train_assets(
    train_part: pd.DataFrame,
    special_tournaments,
    model_families,
):
    assets = {
        "global": {},
        "rare": {},
        "domains": {},
    }

    # Global
    for family in model_families:
        bundle = safe_fit_family_bundle(train_part, family)
        if bundle is not None:
            assets["global"][family] = bundle

    # Rare bucket
    rare_train = train_part.loc[~train_part["tournament"].isin(set(special_tournaments))].copy()
    for family in model_families:
        bundle = safe_fit_family_bundle(rare_train, family)
        if bundle is not None:
            assets["rare"][family] = bundle

    # Domain-specific
    for tournament_name in special_tournaments:
        domain_rows = train_part.loc[train_part["tournament"] == tournament_name].copy()
        assets["domains"][tournament_name] = {}
        for family in model_families:
            bundle = safe_fit_family_bundle(domain_rows, family)
            if bundle is not None:
                assets["domains"][tournament_name][family] = bundle

    return assets


def predict_family_with_policy(
    df_part: pd.DataFrame,
    assets: dict,
    family: str,
    special_tournaments,
    rare_policy: str = "global",   # "global" atau "rare"
):
    assert rare_policy in {"global", "rare"}

    if family not in assets["global"]:
        raise ValueError(f"Global model untuk family={family} tidak tersedia.")

    pred_team, pred_opp = predict_bundle(assets["global"][family], df_part)

    # Domain override untuk special tournaments
    for tournament_name in special_tournaments:
        mask = df_part["tournament"] == tournament_name
        if mask.any() and tournament_name in assets["domains"] and family in assets["domains"][tournament_name]:
            pt, po = predict_bundle(assets["domains"][tournament_name][family], df_part.loc[mask].copy())
            pred_team[mask.values] = pt
            pred_opp[mask.values] = po

    # Rare policy
    if rare_policy == "rare" and family in assets["rare"]:
        rare_mask = ~df_part["tournament"].isin(set(special_tournaments))
        if rare_mask.any():
            pt, po = predict_bundle(assets["rare"][family], df_part.loc[rare_mask].copy())
            pred_team[rare_mask.values] = pt
            pred_opp[rare_mask.values] = po

    return pred_team, pred_opp


def generate_candidate_weights(families):
    families = list(families)
    if len(families) == 1:
        return [{families[0]: 1.0}]

    candidates = []

    # single family
    for fam in families:
        candidates.append({f: (1.0 if f == fam else 0.0) for f in families})

    # equal
    candidates.append({f: 1.0 / len(families) for f in families})

    # dominant blends
    if len(families) >= 2:
        for fam in families:
            others = [x for x in families if x != fam]
            remaining = 0.40 / max(1, len(others))
            w = {f: remaining for f in families}
            w[fam] = 0.60
            candidates.append(w)

    # pair blends when >=3 families
    if len(families) >= 3:
        fams = families
        for i in range(len(fams)):
            for j in range(i + 1, len(fams)):
                w = {f: 0.0 for f in fams}
                w[fams[i]] = 0.50
                w[fams[j]] = 0.50
                candidates.append(w)

    # deduplicate
    seen = set()
    unique_candidates = []
    for cand in candidates:
        key = tuple((k, round(v, 6)) for k, v in sorted(cand.items()))
        if key not in seen:
            seen.add(key)
            unique_candidates.append(cand)

    return unique_candidates


def blend_from_pred_bank(
    pred_bank: dict,
    df_part: pd.DataFrame,
    bucket_weight_map: dict,
    default_weights: dict | None = None,
):
    families = list(pred_bank.keys())
    if default_weights is None:
        default_weights = {f: 1.0 / len(families) for f in families}

    pred_team = np.zeros(len(df_part), dtype=float)
    pred_opp = np.zeros(len(df_part), dtype=float)

    for bucket_name, idx in df_part.groupby("tournament_bucket").groups.items():
        weights = bucket_weight_map.get(bucket_name, default_weights)
        team_bucket = np.zeros(len(idx), dtype=float)
        opp_bucket = np.zeros(len(idx), dtype=float)

        for fam in families:
            w = float(weights.get(fam, 0.0))
            team_bucket += w * pred_bank[fam]["team"][idx]
            opp_bucket += w * pred_bank[fam]["opp"][idx]

        pred_team[idx] = team_bucket
        pred_opp[idx] = opp_bucket

    return pred_team, pred_opp


def tune_bucket_weights(
    df_part: pd.DataFrame,
    pred_bank: dict,
):
    families = list(pred_bank.keys())
    candidates = generate_candidate_weights(families)
    default_weights = {f: 1.0 / len(families) for f in families}

    bucket_weight_map = {}
    tuning_rows = []

    for bucket_name in df_part["tournament_bucket"].dropna().unique().tolist():
        bucket_rows = df_part.loc[df_part["tournament_bucket"] == bucket_name].copy()

        best_metrics = None
        best_match_df_pp = None
        best_weights = None

        for cand in candidates:
            pt = np.zeros(len(bucket_rows), dtype=float)
            po = np.zeros(len(bucket_rows), dtype=float)

            for fam in families:
                w = float(cand.get(fam, 0.0))
                pt += w * pred_bank[fam]["team"][bucket_rows.index]
                po += w * pred_bank[fam]["opp"][bucket_rows.index]

            _, match_df_pp, metrics = metrics_from_row_predictions(bucket_rows, pt, po)

            if (best_metrics is None) or (metrics["awmae"] < best_metrics["awmae"]):
                best_metrics = metrics
                best_match_df_pp = match_df_pp
                best_weights = cand

        if best_weights is None:
            best_weights = default_weights

        bucket_weight_map[bucket_name] = best_weights
        tuning_rows.append({
            "bucket": bucket_name,
            "n_rows": len(bucket_rows),
            "n_matches": bucket_rows["match_id"].nunique(),
            "best_weights": json.dumps(best_weights),
            "awmae": best_metrics["awmae"],
            "base_mae": best_metrics["base_mae"],
            "exact_rate": best_metrics["exact_rate"],
            "outcome_acc": best_metrics["outcome_acc"],
            "gd_acc": best_metrics["gd_acc"],
        })

    tuning_df = pd.DataFrame(tuning_rows).sort_values("awmae", ascending=True).reset_index(drop=True)
    return bucket_weight_map, tuning_df


def tune_postprocessing(df_rows: pd.DataFrame, pred_team_cont: np.ndarray, pred_opp_cont: np.ndarray):
    rows = []
    best = None

    for total_scale in POSTPROCESS_GRID["total_scale"]:
        for draw_shrink in POSTPROCESS_GRID["draw_shrink"]:
            _, match_df_pp, metrics = metrics_from_row_predictions(
                df_rows=df_rows,
                pred_team_cont=pred_team_cont,
                pred_opp_cont=pred_opp_cont,
                total_scale=total_scale,
                draw_shrink=draw_shrink,
            )
            row = {
                "total_scale": total_scale,
                "draw_shrink": draw_shrink,
                **metrics,
            }
            rows.append(row)

            if (best is None) or (metrics["awmae"] < best["metrics"]["awmae"]):
                best = {
                    "params": {"total_scale": total_scale, "draw_shrink": draw_shrink},
                    "match_df_pp": match_df_pp,
                    "metrics": metrics,
                }

    return pd.DataFrame(rows).sort_values("awmae", ascending=True).reset_index(drop=True), best


def build_validation_strategy_row(
    strategy_name: str,
    df_rows: pd.DataFrame,
    pred_team_cont: np.ndarray,
    pred_opp_cont: np.ndarray,
):
    _, match_df_pp, metrics = metrics_from_row_predictions(df_rows, pred_team_cont, pred_opp_cont)
    return {
        "strategy": strategy_name,
        **metrics,
    }, match_df_pp


def build_match_level_submission(
    df_rows: pd.DataFrame,
    pred_team_cont: np.ndarray,
    pred_opp_cont: np.ndarray,
    total_scale: float,
    draw_shrink: float,
):
    match_df = rows_to_canonical_match_df(
        df_rows=df_rows,
        pred_team_cont=pred_team_cont,
        pred_opp_cont=pred_opp_cont,
        include_truth=False,
    )
    match_df_pp = postprocess_match_predictions(
        match_df=match_df,
        total_scale=total_scale,
        draw_shrink=draw_shrink,
    )
    return match_df_pp


def broadcast_match_predictions_back_to_rows(
    df_rows: pd.DataFrame,
    match_pred_df: pd.DataFrame,
):
    temp = build_side_index(df_rows)

    long_records = []
    for _, row in match_pred_df.iterrows():
        long_records.append({
            "match_id": row["match_id"],
            "side_idx": 0,
            "team_goals_pred": int(row["team_goals_pred"]),
            "opp_goals_pred": int(row["opp_goals_pred"]),
        })
        long_records.append({
            "match_id": row["match_id"],
            "side_idx": 1,
            "team_goals_pred": int(row["opp_goals_pred"]),
            "opp_goals_pred": int(row["team_goals_pred"]),
        })

    long_pred = pd.DataFrame(long_records)
    out = temp.merge(long_pred, on=["match_id", "side_idx"], how="left")

    submission = (
        out[["Id", "team_goals_pred", "opp_goals_pred"]]
        .rename(columns={"team_goals_pred": "team_goals", "opp_goals_pred": "opp_goals"})
        .copy()
    )

    submission["team_goals"] = submission["team_goals"].fillna(0).astype(int).clip(lower=0)
    submission["opp_goals"] = submission["opp_goals"].fillna(0).astype(int).clip(lower=0)

    # kembalikan ke urutan test asli
    submission = test_df_raw[["Id"]].merge(submission, on="Id", how="left")
    return submission

# 07. Train Global, Rare, dan Domain-Specific Models

Level model yang disiapkan pada fold validation:

1. **Global model**
2. **OTHER_RARE model**
3. **Special tournament models** untuk tournament yang memenuhi threshold

Setiap family model dilatih **terpisah untuk masing-masing target**:

- `team_goals`
- `opp_goals`

In [ ]:
assets_fold = train_assets(
    train_part=train_fold,
    special_tournaments=special_tournaments_fold,
    model_families=available_model_families,
)

print("Global assets:", list(assets_fold["global"].keys()))
print("Rare assets  :", list(assets_fold["rare"].keys()))
print("Domain assets summary:")
for domain_name, domain_models in assets_fold["domains"].items():
    print(f"- {domain_name}: {list(domain_models.keys())}")

# 08. Validation Comparison — Single Models dan Domain Policies

Kita bandingkan beberapa strategi dasar terlebih dahulu:

- global single model
- per-domain single model + global fallback
- per-domain single model + rare fallback

In [ ]:
validation_results = []
validation_match_predictions = {}

# Global single-family baselines
global_pred_bank = {}
for family in available_model_families:
    if family in assets_fold["global"]:
        pt, po = predict_bundle(assets_fold["global"][family], valid_fold)
        global_pred_bank[family] = {"team": pt, "opp": po}

        row, match_df_pp = build_validation_strategy_row(
            strategy_name=f"global_{family}",
            df_rows=valid_fold,
            pred_team_cont=pt,
            pred_opp_cont=po,
        )
        validation_results.append(row)
        validation_match_predictions[f"global_{family}"] = match_df_pp

# Domain single-family + fallback
policy_pred_banks = {"global": {}, "rare": {}}

for rare_policy in ["global", "rare"]:
    for family in available_model_families:
        if family not in assets_fold["global"]:
            continue

        pt, po = predict_family_with_policy(
            df_part=valid_fold,
            assets=assets_fold,
            family=family,
            special_tournaments=special_tournaments_fold,
            rare_policy=rare_policy,
        )

        policy_pred_banks[rare_policy][family] = {"team": pt, "opp": po}

        row, match_df_pp = build_validation_strategy_row(
            strategy_name=f"domain_{family}_{rare_policy}_fallback",
            df_rows=valid_fold,
            pred_team_cont=pt,
            pred_opp_cont=po,
        )
        validation_results.append(row)
        validation_match_predictions[f"domain_{family}_{rare_policy}_fallback"] = match_df_pp

validation_results_df = pd.DataFrame(validation_results).sort_values("awmae", ascending=True).reset_index(drop=True)
display(validation_results_df)

# 09. Strong Ensemble Per Domain Tournament

Setelah prediksi single-family tersedia, kita lakukan:

1. tuning bobot blend **per bucket tournament**,
2. pembandingan:
   - ensemble dengan **global fallback**
   - ensemble dengan **rare fallback**
3. pemilihan varian ensemble terbaik berdasarkan **AW-MAE validation**

In [ ]:
ensemble_eval_rows = []
ensemble_artifacts = {}

for rare_policy in ["global", "rare"]:
    pred_bank = policy_pred_banks[rare_policy]

    if len(pred_bank) == 0:
        continue

    bucket_weight_map, bucket_tuning_df = tune_bucket_weights(valid_fold, pred_bank)

    default_weights = {f: 1.0 / len(pred_bank) for f in pred_bank.keys()}
    blend_team, blend_opp = blend_from_pred_bank(
        pred_bank=pred_bank,
        df_part=valid_fold,
        bucket_weight_map=bucket_weight_map,
        default_weights=default_weights,
    )

    row, match_df_pp = build_validation_strategy_row(
        strategy_name=f"domain_ensemble_{rare_policy}_fallback",
        df_rows=valid_fold,
        pred_team_cont=blend_team,
        pred_opp_cont=blend_opp,
    )
    validation_results.append(row)
    validation_match_predictions[f"domain_ensemble_{rare_policy}_fallback"] = match_df_pp

    ensemble_eval_rows.append(row)
    ensemble_artifacts[rare_policy] = {
        "pred_team_cont": blend_team,
        "pred_opp_cont": blend_opp,
        "bucket_weight_map": bucket_weight_map,
        "bucket_tuning_df": bucket_tuning_df,
        "default_weights": default_weights,
        "metrics": row,
    }

validation_results_df = pd.DataFrame(validation_results).sort_values("awmae", ascending=True).reset_index(drop=True)
display(validation_results_df)

In [ ]:
print("Tuning bobot blend per bucket (global fallback):")
if "global" in ensemble_artifacts:
    display(ensemble_artifacts["global"]["bucket_tuning_df"])

print("Tuning bobot blend per bucket (rare fallback):")
if "rare" in ensemble_artifacts:
    display(ensemble_artifacts["rare"]["bucket_tuning_df"])

# 10. Pilih Varian Ensemble Final, Lalu Tune Post-Processing

Eksperimen ini memang ditujukan sebagai **strong ensemble experiment**,
jadi strategi final dipilih dari **varian ensemble** yang tersedia.

Setelah itu dilakukan tuning post-processing ringan berbasis validation:

- `rounding`
- `clipping >= 0`
- `total_scale`
- `draw_shrink`

In [ ]:
ensemble_candidates = []
for rare_policy, artifact in ensemble_artifacts.items():
    ensemble_candidates.append({
        "strategy": f"domain_ensemble_{rare_policy}_fallback",
        "rare_policy": rare_policy,
        "awmae": artifact["metrics"]["awmae"],
    })

ensemble_candidates_df = pd.DataFrame(ensemble_candidates).sort_values("awmae", ascending=True).reset_index(drop=True)
display(ensemble_candidates_df)

assert len(ensemble_candidates_df) > 0, "Tidak ada ensemble yang berhasil dibangun."

selected_ensemble_strategy = ensemble_candidates_df.iloc[0]["strategy"]
selected_rare_policy = ensemble_candidates_df.iloc[0]["rare_policy"]
selected_ensemble_artifact = ensemble_artifacts[selected_rare_policy]

print("Selected ensemble strategy:", selected_ensemble_strategy)
print("Selected rare policy      :", selected_rare_policy)

postprocess_results_df, best_postprocess = tune_postprocessing(
    df_rows=valid_fold,
    pred_team_cont=selected_ensemble_artifact["pred_team_cont"],
    pred_opp_cont=selected_ensemble_artifact["pred_opp_cont"],
)

display(postprocess_results_df)

selected_postprocess_params = best_postprocess["params"]
selected_valid_match_df_pp = best_postprocess["match_df_pp"]
selected_valid_metrics = best_postprocess["metrics"]

print("Selected post-processing params:", selected_postprocess_params)
print("Validation metrics after selected post-processing:")
display(pd.DataFrame([selected_valid_metrics]))

In [ ]:
# Perbandingan sebelum vs sesudah post-processing untuk ensemble terpilih
_, selected_match_df_default_pp, default_metrics = metrics_from_row_predictions(
    valid_fold,
    selected_ensemble_artifact["pred_team_cont"],
    selected_ensemble_artifact["pred_opp_cont"],
    total_scale=1.0,
    draw_shrink=0.0,
)

before_after_df = pd.DataFrame([
    {"stage": "before_postprocessing", **default_metrics},
    {"stage": "after_postprocessing", **selected_valid_metrics},
]).sort_values("awmae", ascending=True)

display(before_after_df)

compare_pp = selected_match_df_default_pp[[
    "match_id", "tournament", "team_goals_true", "opp_goals_true", "team_goals_pred", "opp_goals_pred"
]].rename(columns={
    "team_goals_pred": "pred_team_before_pp",
    "opp_goals_pred": "pred_opp_before_pp",
}).merge(
    selected_valid_match_df_pp[[
        "match_id", "team_goals_pred", "opp_goals_pred"
    ]].rename(columns={
        "team_goals_pred": "pred_team_after_pp",
        "opp_goals_pred": "pred_opp_after_pp",
    }),
    on="match_id",
    how="left"
)

changed_after_pp = compare_pp.loc[
    (compare_pp["pred_team_before_pp"] != compare_pp["pred_team_after_pp"])
    | (compare_pp["pred_opp_before_pp"] != compare_pp["pred_opp_after_pp"])
].copy()

print("Contoh prediksi yang berubah setelah post-processing:")
display(changed_after_pp.head(20))

# 11. Analisis Hasil Validation

Bagian ini menampilkan:

- tabel prediksi vs aktual,
- error analysis singkat,
- ringkasan domain tournament,
- indikasi apakah ensemble membantu,
- indikasi apakah rare bucket membantu,
- indikasi apakah post-processing membantu.

In [ ]:
print("Tabel hasil prediksi vs aktual (validation, match-level):")
validation_preview = selected_valid_match_df_pp[[
    "match_id", "tournament", "team_goals_true", "opp_goals_true", "team_goals_pred", "opp_goals_pred"
]].head(25)
display(validation_preview)

loss_table = match_loss_table(selected_valid_match_df_pp)
print("Error analysis singkat — match dengan weighted loss terbesar:")
display(
    loss_table.sort_values("weighted_loss", ascending=False)[[
        "match_id", "tournament",
        "team_goals_true", "opp_goals_true",
        "team_goals_pred", "opp_goals_pred",
        "match_base_mae", "exact", "outcome_correct", "gd_correct",
        "weight", "weighted_loss"
    ]].head(20)
)

In [ ]:
# Ringkasan performa per domain tournament pada validation
valid_perf_by_tournament = (
    match_loss_table(selected_valid_match_df_pp)
    .groupby("tournament", as_index=False)
    .agg(
        n_matches=("match_id", "count"),
        avg_base_mae=("match_base_mae", "mean"),
        exact_rate=("exact", "mean"),
        outcome_acc=("outcome_correct", "mean"),
        gd_acc=("gd_correct", "mean"),
        avg_weighted_loss=("weighted_loss", "mean"),
    )
    .sort_values("avg_weighted_loss", ascending=False)
    .reset_index(drop=True)
)

print("Ringkasan performa validation per tournament:")
display(valid_perf_by_tournament.head(25))

In [ ]:
# Ringkasan domain tournament yang dipakai untuk modeling pada fold
domain_modeling_stats = (
    pd.DataFrame({"tournament": train_fold["tournament"].fillna("MISSING").value_counts().index})
    .merge(
        train_fold["tournament"].fillna("MISSING").value_counts().rename("train_rows").reset_index().rename(columns={"index": "tournament"}),
        on="tournament",
        how="left",
    )
    .merge(
        valid_fold["tournament"].fillna("MISSING").value_counts().rename("valid_rows").reset_index().rename(columns={"index": "tournament"}),
        on="tournament",
        how="left",
    )
    .fillna(0)
)

domain_modeling_stats["train_rows"] = domain_modeling_stats["train_rows"].astype(int)
domain_modeling_stats["valid_rows"] = domain_modeling_stats["valid_rows"].astype(int)
domain_modeling_stats["uses_special_model"] = domain_modeling_stats["tournament"].isin(special_tournaments_fold)
domain_modeling_stats["bucket"] = np.where(
    domain_modeling_stats["uses_special_model"],
    domain_modeling_stats["tournament"],
    "OTHER_RARE",
)
domain_modeling_stats["fallback_policy_selected"] = np.where(
    domain_modeling_stats["uses_special_model"],
    "domain_specific",
    selected_rare_policy,
)

display(
    domain_modeling_stats.sort_values(
        ["uses_special_model", "train_rows"], ascending=[False, False]
    ).head(40)
)

In [ ]:
# Analisis apakah ensemble membantu dibanding model tunggal terbaik
single_model_rows = validation_results_df.loc[
    ~validation_results_df["strategy"].str.contains("ensemble")
].copy()

best_single_model_row = single_model_rows.sort_values("awmae", ascending=True).head(1)
best_ensemble_model_row = validation_results_df.loc[
    validation_results_df["strategy"] == selected_ensemble_strategy
].copy()

comparison_best_single_vs_ensemble = pd.concat([
    best_single_model_row.assign(kind="best_single_model"),
    best_ensemble_model_row.assign(kind="selected_ensemble"),
], axis=0)

display(comparison_best_single_vs_ensemble)

In [ ]:
# Feature importance (jika model global yang dipilih mendukung)
# Ini hanya analisis diagnostik, bukan penentu utama final selection.
family_for_importance = None
for fam in ["cat", "lgb", "xgb"]:
    if fam in assets_fold["global"]:
        family_for_importance = fam
        break

if family_for_importance is not None:
    bundle = assets_fold["global"][family_for_importance]
    model = bundle["team_model"]
    feature_names = bundle["feature_names_after_preprocess"]

    fi = None
    if hasattr(model, "feature_importances_"):
        fi = model.feature_importances_
    elif hasattr(model, "get_feature_importance"):
        fi = model.get_feature_importance()

    if fi is not None:
        fi_df = pd.DataFrame({
            "feature": feature_names,
            "importance": np.asarray(fi, dtype=float),
        }).sort_values("importance", ascending=False).reset_index(drop=True)

        print(f"Feature importance (global {family_for_importance}, target=team_goals):")
        display(fi_df.head(20))
    else:
        print("Model tidak menyediakan feature importance yang mudah diambil.")
else:
    print("Tidak ada global model yang berhasil dilatih untuk analisis feature importance.")

# 12. Retrain pada Full Train dan Final Test Inference

Setelah strategi final dipilih dari validation:

1. tentukan ulang special tournaments dari **full train**
2. latih ulang global / rare / special-domain models
3. lakukan prediksi `test.csv`
4. blend sesuai bobot bucket yang dipilih
5. terapkan post-processing final
6. simpan submission ke:
   **`exp08_tournament_specific_strong_ensemble.csv`**

In [ ]:
# Special tournaments final dari full train
special_tournaments_full, tournament_counts_full = get_special_tournaments(
    train_df,
    min_domain_rows=MIN_DOMAIN_ROWS,
    max_special_tournaments=MAX_SPECIAL_TOURNAMENTS,
)

train_df["tournament_bucket"] = assign_tournament_bucket(train_df["tournament"], special_tournaments_full)
test_df["tournament_bucket"] = assign_tournament_bucket(test_df["tournament"], special_tournaments_full)

print("Special tournaments (full train):")
print(special_tournaments_full)

assets_full = train_assets(
    train_part=train_df,
    special_tournaments=special_tournaments_full,
    model_families=available_model_families,
)

print("Full-train global assets:", list(assets_full["global"].keys()))
print("Full-train rare assets  :", list(assets_full["rare"].keys()))

In [ ]:
# Bangun prediksi test per family dengan rare policy yang sudah dipilih
test_pred_bank = {}

for family in available_model_families:
    if family not in assets_full["global"]:
        continue

    pt, po = predict_family_with_policy(
        df_part=test_df,
        assets=assets_full,
        family=family,
        special_tournaments=special_tournaments_full,
        rare_policy=selected_rare_policy,
    )
    test_pred_bank[family] = {"team": pt, "opp": po}

assert len(test_pred_bank) > 0, "Tidak ada model family yang tersedia untuk test inference."

In [ ]:
# Terapkan bobot blend yang dituning dari validation.
# Jika ada special tournament baru di full train tetapi bucket weight belum pernah dituning di validation,
# gunakan default weights (equal blend).
selected_bucket_weight_map = selected_ensemble_artifact["bucket_weight_map"]
selected_default_weights = selected_ensemble_artifact["default_weights"]

test_blend_team_cont, test_blend_opp_cont = blend_from_pred_bank(
    pred_bank=test_pred_bank,
    df_part=test_df,
    bucket_weight_map=selected_bucket_weight_map,
    default_weights=selected_default_weights,
)

test_match_pred_df = build_match_level_submission(
    df_rows=test_df,
    pred_team_cont=test_blend_team_cont,
    pred_opp_cont=test_blend_opp_cont,
    total_scale=selected_postprocess_params["total_scale"],
    draw_shrink=selected_postprocess_params["draw_shrink"],
)

submission_df = broadcast_match_predictions_back_to_rows(
    df_rows=test_df,
    match_pred_df=test_match_pred_df,
)

# Format final wajib persis
submission_df = submission_df[["Id", "team_goals", "opp_goals"]].copy()
submission_df["team_goals"] = submission_df["team_goals"].astype(int).clip(lower=0)
submission_df["opp_goals"] = submission_df["opp_goals"].astype(int).clip(lower=0)

assert list(submission_df.columns) == ["Id", "team_goals", "opp_goals"]
assert len(submission_df) == len(test_df_raw)

display(submission_df.head(20))

In [ ]:
submission_df.to_csv(FINAL_CSV_NAME, index=False)
print(f"Submission saved to: {FINAL_CSV_NAME}")
print("Final submission shape:", submission_df.shape)

# 13. Conclusion

Notebook ini sudah mencakup seluruh komponen inti yang diminta:

- training pipeline row-wise
- validation leakage-safe berbasis `date` + `match_id`
- evaluator resmi **AW-MAE**
- prediksi validation dan tabel prediksi vs aktual
- ringkasan fitur yang dipakai
- ringkasan domain split per `tournament`
- strong ensemble antar model tabular
- post-processing berbasis AW-MAE
- final inference pada `test.csv`
- pembuatan file submission:
  **`exp08_tournament_specific_strong_ensemble.csv`**

## Ringkasan interpretasi hasil
Saat notebook dijalankan:
1. lihat tabel `validation_results_df` untuk membandingkan global vs domain vs ensemble,
2. lihat `before_after_df` untuk mengecek apakah post-processing membantu,
3. lihat `valid_perf_by_tournament` untuk mengidentifikasi domain yang paling terbantu,
4. lihat `domain_modeling_stats` untuk memastikan fallback policy konsisten.

In [ ]:
# Ringkasan singkat objek final yang paling penting
final_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "final_csv_name": FINAL_CSV_NAME,
    "n_feature_cols": len(feature_cols),
    "selected_ensemble_strategy": selected_ensemble_strategy,
    "selected_rare_policy": selected_rare_policy,
    "selected_postprocess_params": selected_postprocess_params,
    "special_tournaments_fold": special_tournaments_fold,
    "special_tournaments_full": special_tournaments_full,
}
display(pd.DataFrame([final_summary]))